# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [83]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.metrics import classification_report, accuracy_score


In [84]:
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


In [85]:
# La columna de estudio es: 

# fraud → 0 = normal, 1 = fraude

**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?


In [86]:
fraud['fraud'].value_counts(normalize=True)

fraud
0.0    0.912597
1.0    0.087403
Name: proportion, dtype: float64

In [87]:
fraud['fraud'].value_counts() # conteos absolutos


fraud
0.0    912597
1.0     87403
Name: count, dtype: int64

El dataset está muy desbalanceado y esto destruye cualquier modelo, El modelo ignora la clase minoritaria

La métrica accuracy deja de ser válida, acierta los no fraudes pero no te garantiza acertar los fraudes

Con un 8.7% de fraudes, un modelo trivial que prediga siempre “no fraude” tendría 91% de Accuracy, pero sería inútil.


El modelo se vuelve inútil para el objetivo real (detectar fraudes)



- **2.** Train a LogisticRegression.


In [88]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv"
df = pd.read_csv(url)

# Separa features y target
X = df.drop("fraud", axis=1)
y = df["fraud"]

# 3. Train-test split (stratify=y asegura que el desbalance se mantiene igual en ambos. REVISAR
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Scaling --> Logistic Regression necesita que todas las variables estén en la misma escala.
# vip: Logistic Regression → clasifica 0/1 usando una frontera lineal.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. Logistic Regression model -->  Entrenas el modelo con los datos desbalanceados.
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_scaled, y_train)

# 6. Predictions
y_pred = log_reg.predict(X_test_scaled)

# 7. Evaluation
print(classification_report(y_test, y_pred))
print("F1-score (fraud class):", f1_score(y_test, y_pred, average='binary'))


              precision    recall  f1-score   support

         0.0       0.96      0.99      0.98    182519
         1.0       0.90      0.61      0.72     17481

    accuracy                           0.96    200000
   macro avg       0.93      0.80      0.85    200000
weighted avg       0.96      0.96      0.96    200000

F1-score (fraud class): 0.7228405599180607


- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.


In [89]:

# - Accuracy NO es válida por el desbalance (91% vs 9%).
# - La métrica correcta es F1 de la clase 1 (fraude).
# - El modelo baseline (LogicalRegression) tiene bajo F1 en fraude → no detecta bien los casos importantes.

# Precision (fraude)
# De todas las transacciones que el modelo marca como fraude, ¿cuántas lo son realmente?
    # Precision alta → pocos falsos positivos --> esto esta bien pero no es lo que mas nos importa en el estudio
    # por que un falso positivo molesta al cliente, pero se puede revisar manualmente.

# Recall (fraude)
    # De todos los fraudes reales, ¿cuántos detecta el modelo?
    # Recall alto → el modelo detecta la mayoría de fraudes
    # Recall bajo → el modelo deja escapar fraudes reales

# Esta es la metrica mas importante para ver si funciona bien , pero matematicamente la que manda es F1
#  que busca recall y precision altos

- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 


In [90]:
# Haces que haya tantos fraudes como no fraudes duplicando los fraudes.
from  imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import classification_report, f1_score

# 1. Oversampling SOLO en el set de entrenamiento
ros = RandomOverSampler(random_state=42)
X_train_ros, y_train_ros = ros.fit_resample(X_train, y_train)

# 2. Escalado
X_train_ros_scaled = scaler.fit_transform(X_train_ros)
X_test_scaled = scaler.transform(X_test)

# 3. Entrenar Logistic Regression con datos balanceados
log_reg_ros = LogisticRegression(max_iter=1000)
log_reg_ros.fit(X_train_ros_scaled, y_train_ros)

# 4. Predicciones
y_pred_ros = log_reg_ros.predict(X_test_scaled)

# 5. Evaluación
print("=== Classification Report (Oversampling) ===")
print(classification_report(y_test, y_pred_ros))

f1_fraud_ros = f1_score(y_test, y_pred_ros, average='binary')
print("F1-score (fraud class) with Oversampling:", f1_fraud_ros)


=== Classification Report (Oversampling) ===
              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182519
         1.0       0.58      0.95      0.72     17481

    accuracy                           0.93    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000

F1-score (fraud class) with Oversampling: 0.7175555988652851


In [91]:
# Puntuaciones_Baseline: 
#               precision    recall  f1-score   support

#          0.0       0.96      0.99      0.98    182519
#          1.0       0.90      0.61      0.72     17481

#     accuracy                           0.96    200000
#    macro avg       0.93      0.80      0.85    200000
# weighted avg       0.96      0.96      0.96    200000

# F1-score (fraud class): 0.7228405599180607

In [92]:
# El Oversampling no ha mejorado el F1 por Overfitting 
# LogisticalRegression es un modelo lineal, necesita variacion de datos no datos repetidos
# Mejora muchisimo el recall, Detecta casi todos los fraudes
# pero se equivoca muchisimimo baja la Precision muchisimo de hay que F1 se mantenga

- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?


In [93]:
# Undersampling (borrar normales)
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import classification_report, f1_score

# 1. Undersampling SOLO en el set de entrenamiento
rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

# 2. Escalado
X_train_rus_scaled = scaler.fit_transform(X_train_rus)
X_test_scaled = scaler.transform(X_test)

# 3. Entrenar Logistic Regression con datos reducidos
log_reg_rus = LogisticRegression(max_iter=1000)
log_reg_rus.fit(X_train_rus_scaled, y_train_rus)

# 4. Predicciones
y_pred_rus = log_reg_rus.predict(X_test_scaled)

# 5. Evaluación
print("=== Classification Report (Undersampling) ===")
print(classification_report(y_test, y_pred_rus))

f1_fraud_rus = f1_score(y_test, y_pred_rus, average='binary')
print("F1-score (fraud class) with Undersampling:", f1_fraud_rus)


=== Classification Report (Undersampling) ===
              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182519
         1.0       0.58      0.95      0.72     17481

    accuracy                           0.93    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000

F1-score (fraud class) with Undersampling: 0.7174478038638136


In [94]:
# Puntuaciones_Baseline: 
#               precision    recall  f1-score   support

#          0.0       0.96      0.99      0.98    182519
#          1.0       0.90      0.61      0.72     17481

#     accuracy                           0.96    200000
#    macro avg       0.93      0.80      0.85    200000
# weighted avg       0.96      0.96      0.96    200000

# F1-score (fraud class): 0.7228405599180607

In [95]:
# Mismos resultados que el Oversampling. Sigue habiendo ese desequilibrio entre PrecissiON y Recall que hace que F1 no suba.
# Tanto subiendo los casos falso como bajando los positivos para hacer el balanceo vemos que da el mismo resultado.

- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

In [96]:
# SMOTE (crear fraudes sintéticos)
# SMOTE crea nuevos fraudes mezclando los existentes.
# No duplica, no borra → genera datos más realistas
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, f1_score

# 1. SMOTE SOLO en el set de entrenamiento
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# 2. Escalado
X_train_smote_scaled = scaler.fit_transform(X_train_smote)
X_test_scaled = scaler.transform(X_test)

# 3. Entrenar Logistic Regression con datos sintéticos balanceados
log_reg_smote = LogisticRegression(max_iter=1000)
log_reg_smote.fit(X_train_smote_scaled, y_train_smote)

# 4. Predicciones
y_pred_smote = log_reg_smote.predict(X_test_scaled)

# 5. Evaluación
print("=== Classification Report (SMOTE) ===")
print(classification_report(y_test, y_pred_smote))

f1_fraud_smote = f1_score(y_test, y_pred_smote, average='binary')
print("F1-score (fraud class) with SMOTE:", f1_fraud_smote)


=== Classification Report (SMOTE) ===
              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182519
         1.0       0.58      0.95      0.72     17481

    accuracy                           0.94    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.94      0.94    200000

F1-score (fraud class) with SMOTE: 0.7184398549308315


In [97]:
# Puntuaciones_Baseline: 
#               precision    recall  f1-score   support

#          0.0       0.96      0.99      0.98    182519
#          1.0       0.90      0.61      0.72     17481

#     accuracy                           0.96    200000
#    macro avg       0.93      0.80      0.85    200000
# weighted avg       0.96      0.96      0.96    200000

# F1-score (fraud class): 0.7228405599180607

In [98]:
# El modelo detecta aún más fraudes

# El F1-score de fraude es el mejor de todos

# No pierde información

# Es la técnica más equilibrada

In [102]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

def evaluar_modelo_completo(nombre, modelo, X_test, y_test):
    y_pred = modelo.predict(X_test)

    # Métricas para clase 0
    precision_0 = precision_score(y_test, y_pred, pos_label=0)
    recall_0    = recall_score(y_test, y_pred, pos_label=0)
    f1_0        = f1_score(y_test, y_pred, pos_label=0)

    # Métricas para clase 1 (fraude)
    precision_1 = precision_score(y_test, y_pred, pos_label=1)
    recall_1    = recall_score(y_test, y_pred, pos_label=1)
    f1_1        = f1_score(y_test, y_pred, pos_label=1)

    acc = accuracy_score(y_test, y_pred)

    return {
        "Modelo": nombre,
        "Precision_0": precision_0,
        "Recall_0": recall_0,
        "F1_0": f1_0,
        "Precision_1": precision_1,
        "Recall_1": recall_1,
        "F1_1": f1_1,
        "Accuracy": acc
    }



In [103]:
resultados = []

resultados.append(evaluar_modelo_completo("Baseline", log_reg, X_test_scaled, y_test))
resultados.append(evaluar_modelo_completo("Oversampling", log_reg_ros, X_test_scaled, y_test))
resultados.append(evaluar_modelo_completo("Undersampling", log_reg_rus, X_test_scaled, y_test))
resultados.append(evaluar_modelo_completo("SMOTE", log_reg_smote, X_test_scaled, y_test))



In [106]:
df_resultados = pd.DataFrame(resultados)
df_resultados

# NOTA 
# PRECISION_0 = precisión de la clase 0 (NO fraude)

# PRECISION_1 = precisión de la clase 1 (FRAUDE)

# No suman 100 porque no son partes del mismo total.
# Cada una se calcula sobre predicciones distintas.

,Modelo,Precision_0,Recall_0,F1_0,Precision_1,Recall_1,F1_1,Accuracy
0,Baseline,0.921879,0.998975,0.958880,0.915652,0.116126,0.206112,0.921810
1,Oversampling,0.994648,0.933634,0.963176,0.577606,0.947543,0.717709,0.934850
2,Undersampling,0.994629,0.933415,0.963050,0.576757,0.947371,0.717004,0.934635
3,SMOTE,0.994517,0.934116,0.963371,0.579045,0.946227,0.718440,0.935175


In [107]:
# Tiene el mejor F1-score (0.7184)
# F1 es la métrica que equilibra:

# Precision (evitar falsos positivos)

# Recall (detectar fraudes)

# Y SMOTE es el que logra el mejor equilibrio.

# Tiene un Recall altísimo (0.9462)
# Detecta casi TODOS los fraudes reales.

# Tiene una Precision razonable (0.5790)
# No es perfecta, pero es aceptable en fraude.